# N-BEATS predictions export (for the ensemble)

Saves the champion N-BEATS predictions so the ensemble can use the real neural model
(instead of a placeholder). Two files, keyed by Store/Dept/Date so they merge cleanly:

- `predictions/nn_val_preds.csv`: validation predictions from a **train-only** model
  (no leakage), for honest ensemble weight tuning.
- `predictions/nn_test_preds.csv`: test predictions from the **full-history** model,
  for the final Kaggle submission.

Columns: `Store, Dept, Date, nn_pred`.

In [1]:
# Run from the project root so data/ and src/ work.
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: D:\Desktop\ML Project


In [2]:
import yaml
import warnings
import numpy as np
import pandas as pd
from neuralforecast import NeuralForecast

from src.features.nn_preprocessing import build_long_df, temporal_split, get_real_validation, FREQ
from src.features.preprocessing import merge_and_enrich
from src.models.nbeats_pipeline import build_nbeats
from src.evaluation.metrics import calculate_wmae

warnings.filterwarnings("ignore")

with open("configs/nbeats_config.yaml") as f:
    config = yaml.safe_load(f)
params = config["model"]["params"]
split_date = config["data"]["split_date"]
os.makedirs("predictions", exist_ok=True)

D:\Desktop\ML Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-11 20:01:46,753	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-07-11 20:01:47,043	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [3]:
train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")
test_raw = pd.read_csv("data/test.csv")

Y_df = build_long_df(train_raw, stores, features, fill_gaps=True)
train_df, valid_df, horizon = temporal_split(Y_df, split_date)
real_valid = get_real_validation(train_raw, stores, features, split_date)
print("horizon:", horizon)

horizon: 43


## Validation predictions (train-only model, no leakage)

In [4]:
# Train N-BEATS on the training period only, then forecast the validation weeks.
nf_val = build_nbeats(horizon, params, freq=FREQ)
nf_val.fit(df=train_df[["unique_id", "ds", "y"]])
fcst_val = nf_val.predict()

# Build the real validation rows with their keys and attach the forecast.
val_keys = real_valid[["unique_id", "ds"]].copy()
val_keys["Store"] = val_keys["unique_id"].str.split("_").str[0].astype(int)
val_keys["Dept"] = val_keys["unique_id"].str.split("_").str[1].astype(int)
val_out = val_keys.merge(fcst_val[["unique_id", "ds", "NBEATS"]], on=["unique_id", "ds"], how="left")
val_out["nn_pred"] = val_out["NBEATS"].fillna(0).clip(lower=0)
val_out["Date"] = val_out["ds"].dt.strftime("%Y-%m-%d")

val_export = val_out[["Store", "Dept", "Date", "nn_pred"]]
val_export.to_csv("predictions/nn_val_preds.csv", index=False)

# Sanity check: WMAE of these predictions on the real validation rows.
wmae = calculate_wmae(real_valid["y"].values, val_out["nn_pred"].values, real_valid["IsHoliday"].values)
print("Saved predictions/nn_val_preds.csv | rows:", len(val_export), "| N-BEATS val WMAE:", round(wmae, 2))

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 5.2 M  | train
-------------------------------------------------------
5.2 M     Trainable params
0         Non-trainable params
5.2 M     Total params
20.732    Total estimated model params size (MB)
58        Modules in train mode
0         Modules in eval mode


Epoch 0:  96%|█████████▌| 100/104 [00:15<00:00,  6.61it/s, v_num=11, train_loss_step=81.10] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1:  92%|█████████▏| 96/104 [00:19<00:01,  4.93it/s, v_num=11, train_loss_step=134.0, train_loss_epoch=109.0]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 2:  88%|████████▊ | 92/104 [00:23<00:03,  3.91it/s, v_num=11, train_loss_step=16.90, train_loss_epoch=117.0]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3:  85%|████████▍ | 88/104 [00:18<00:03,  4.72it/s, v_num=11, train_loss_step=456.0, train_loss_epoch=118.0]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 4:  81%|████████  | 84/104 [00:18<00:04,  4.48it/s, v_num=11, train_loss_step=111.0, train_loss_epoch=140.0]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 

`Trainer.fit` stopped: `max_steps=1000` reached.


Epoch 9: 100%|██████████| 64/64 [00:17<00:00,  3.75it/s, v_num=11, train_loss_step=67.70, train_loss_epoch=113.0]


GPU available: False, used: False
TPU available: False, using: 0 TPU cores


Predicting DataLoader 0: 100%|██████████| 104/104 [00:01<00:00, 84.92it/s]
Saved predictions/nn_val_preds.csv | rows: 127438 | N-BEATS val WMAE: 2274.69


## Test predictions (full-history model, for the submission)

In [5]:
# Load the full-history champion saved by model_experiment_NBEATS.ipynb.
nf_full = NeuralForecast.load(path="saved_models/nbeats_nf")
fcst_test = nf_full.predict(df=Y_df[["unique_id", "ds", "y"]])

test_frame = test_raw[["Store", "Dept", "Date"]].copy()
test_frame["unique_id"] = test_frame["Store"].astype(str) + "_" + test_frame["Dept"].astype(str)
test_frame["ds"] = pd.to_datetime(test_frame["Date"])

test_out = test_frame.merge(fcst_test[["unique_id", "ds", "NBEATS"]], on=["unique_id", "ds"], how="left")
test_out["nn_pred"] = test_out["NBEATS"].fillna(0).clip(lower=0)

test_export = test_out[["Store", "Dept", "Date", "nn_pred"]]
test_export.to_csv("predictions/nn_test_preds.csv", index=False)
print("Saved predictions/nn_test_preds.csv | rows:", len(test_export),
      "| matched:", int(test_out["NBEATS"].notna().sum()))

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores


Predicting DataLoader 0: 100%|██████████| 105/105 [00:01<00:00, 91.06it/s]
Saved predictions/nn_test_preds.csv | rows: 115064 | matched: 114788


## Notes

- The ensemble should merge these files by `[Store, Dept, Date]` (robust, order-
  independent) and use `nn_pred` as the neural component.
- Validation preds come from a train-only model (honest); test preds from the full
  model (for the real submission).